In [9]:
%pip install pandas openpyxl


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd

df = pd.read_excel("incidents_2024_oct2026.xlsx")
df.head()

df = pd.read_excel("incidents_2024_oct2026.xlsx")
df["Neighborhood"] = df["Neighborhood"].str.replace("–", "-", regex=False).str.strip()  ## This is to get rid of the overlapping city names caused by a differenc ein the usage of hyphens and en dashes

In [11]:
print(df.shape)
print(df.columns.tolist())

(116730, 20)
['Report_Number', 'ReportedDate', 'ReportedTime', 'Hour', 'DayofWeek', 'ReportedMonth', 'ReportedYear', 'NIBRS_Coded_Offense', 'NIBRS_Offense_Code', 'NIBRS_Offense_Category', 'NIBRS_Offense_Type', 'NIBRS_Crime_Against', 'NIBRS_Offense_Grouping', 'Violation', 'XCOORD', 'YCOORD', 'Zone', 'Tract', 'Neighborhood', 'Block_Address']


In [12]:
counts = df.groupby("Neighborhood")["Report_Number"].nunique().sort_values()
print(counts.head(10))   # fewest incidents
print(counts.tail(10))   # most incidents

Neighborhood
St. Clair          35
Ridgemont          42
New Homestead      56
Chartiers City     64
East Carnegie      69
Swisshelm Park     84
Summer Hill        90
Fairywood         113
Esplen            120
Hays              122
Name: Report_Number, dtype: int64
Neighborhood
Brookline                    1670
East Allegheny               1705
Mount Washington             1740
Squirrel Hill South          1754
Bloomfield                   2071
East Liberty                 2419
Carrick                      2723
Shadyside                    2808
South Side Flats             4659
Central Business District    7742
Name: Report_Number, dtype: int64


In [13]:
df["Neighborhood"].isna().sum() ## checking to see how many of the entries don't have a neighborhood name 
## uh compared to the amount of entires there are close to 1% don't have a name so I'll mention it in the ending but will ignore it in my math

np.int64(1259)

In [14]:
monthly = (
    df.groupby(["Neighborhood", "ReportedYear", "ReportedMonth"])["Report_Number"]
    .nunique()
    .unstack(["ReportedYear", "ReportedMonth"])
)
monthly.head()

ReportedYear       2024                                                        \
ReportedMonth       Apr   Aug   Dec   Feb   Jan   Jul   Jun   Mar   May   Nov   
Neighborhood                                                                    
Allegheny Center   16.0  19.0  12.0   6.0   8.0  20.0  10.0  14.0  13.0  15.0   
Allegheny West      3.0   4.0   4.0   4.0   3.0   3.0   7.0   NaN   6.0   5.0   
Allentown          32.0  45.0  35.0  24.0  29.0  48.0  41.0  30.0  39.0  28.0   
Arlington          15.0  11.0  13.0  16.0   8.0  10.0  12.0  10.0   6.0   8.0   
Arlington Heights   6.0   6.0   6.0   6.0   3.0   3.0   3.0   2.0   2.0   2.0   

ReportedYear       ...  2026                                                  \
ReportedMonth      ...   Apr   Aug   Feb   Jan   Jul   Jun   Mar   May   Oct   
Neighborhood       ...                                                         
Allegheny Center   ...  13.0  11.0   7.0  12.0  13.0  16.0  14.0   9.0  13.0   
Allegheny West     ...  10.0   6.0  10.0   5.0   4.0   6.0   6.0   1.0   7.0   
Allentown          ...  40.0  39.0  37.0  34.0  35.0  42.0  28.0  37.0  38.0   
Arlington          ...  12.0  15.0  22.0  14.0  10.0  14.0  20.0  23.0  14.0   
Arlington Heights  ...   1.0   4.0   6.0   1.0   6.0  10.0   4.0   5.0   2.0   

ReportedYear             
ReportedMonth       Sep  
Neighborhood             
Allegheny Center   17.0  
Allegheny West      6.0  
Allentown          26.0  
Arlington          18.0  
Arlington Heights   3.0  

[5 rows x 34 columns]

In [15]:
monthly = monthly.fillna(0)

summary = pd.DataFrame({
    "total": monthly.sum(axis=1), ## the amount of reports acroos the table
    "avg_per_month": monthly.mean(axis=1),
    "std_dev": monthly.std(axis=1)
}).sort_values("avg_per_month")

summary.head(10)

,total,avg_per_month,std_dev
Neighborhood,,,
St. Clair,35.0,1.029412,1.086705
Ridgemont,42.0,1.235294,1.304046
New Homestead,56.0,1.647059,1.069759
Chartiers City,64.0,1.882353,1.273618
East Carnegie,70.0,2.058824,1.536252
Swisshelm Park,84.0,2.470588,1.862473
Summer Hill,90.0,2.647059,1.773241
Fairywood,113.0,3.323529,2.170331
Esplen,120.0,3.529412,1.813005


When I first ran this line of code I saw that a few of the 

In [16]:
names = df["Neighborhood"].dropna().unique()
print([n for n in names if "Spring" in n or "Lincoln" in n])

['Lincoln-Lemington-Belmar', 'Spring Hill-City View', 'Spring Garden', 'Lincoln Place']
